# Run LLaMA-Factory as an Anyscale Job

For longer or production runs, submit fine-tuning as an **Anyscale job**. Jobs run outside your interactive session for better stability, retries, and durable logs. You package LLaMA-Factory and other libraries in a container image and launch with a short job config.

This guide walks through the SFT with LoRA and DeepSpeed example as an Anyscale job. You can adapt this approach for any of the other training methods (DPO, KTO, CPT) by swapping the training config.

## Prerequisites

- [Anyscale CLI](https://docs.anyscale.com/reference/quickstart#step-2-install-the-cli) installed and authenticated (`anyscale login`)
- A running [Anyscale workspace](https://docs.anyscale.com/platform/workspaces/) to prepare shared storage files

## Step 1: Build a custom container image

Build this image in the Anyscale console (**Configurations > Container Images > Build**):

```dockerfile
FROM anyscale/ray-llm:2.48.0-py311-cu128
RUN pip install --no-cache-dir \
    "llamafactory==0.9.3" \
    "deepspeed==0.16.9" \
    "wandb==0.21.3"
```

Note the resulting `image_uri` and update `job.yaml` accordingly.

## Step 2: Prepare shared storage

Copy the required files to `/mnt/user_storage/` via a running workspace. You also need to copy and adapt the training config, updating paths from `/mnt/cluster_storage/` to `/mnt/user_storage/`.

In [ ]:
%%bash
# Copy DeepSpeed config
cp ../deepspeed-configs/ds_z3_config.json /mnt/user_storage/

# Copy and adapt dataset registry (update paths from /mnt/cluster_storage/ to /mnt/user_storage/)
sed 's|/mnt/cluster_storage|/mnt/user_storage|g' ../dataset-configs/dataset_info.json > /mnt/user_storage/dataset_info.json

# Download demo dataset
wget https://anyscale-public-materials.s3.us-west-2.amazonaws.com/llm-finetuning/llama-factory/datasets/sharegpt/glaive_toolcall_en_demo.json -O /mnt/user_storage/glaive_toolcall_en_demo.json

# Copy and adapt the training config (update paths from /mnt/cluster_storage/ to /mnt/user_storage/)
sed 's|/mnt/cluster_storage|/mnt/user_storage|g' ../train-configs/sft_lora_deepspeed.yaml > /mnt/user_storage/sft_lora_deepspeed.yaml

## Step 3: Create the job config

In [ ]:
%%bash
cat ../job-configs/job.yaml

## Step 4: Submit and monitor the job

In [ ]:
%%bash
# Submit the job
anyscale job submit -f ../job-configs/job.yaml

# Or submit and wait for completion with log streaming:
# anyscale job submit -f ../job-configs/job.yaml --wait

### Monitor the job

In [ ]:
%%bash
# Check job status
anyscale job status -n qwen2.5_deepspeed_lora_sft_job

# Terminate if needed
# anyscale job terminate -n qwen2.5_deepspeed_lora_sft_job

## Output

Training output (checkpoints, loss plots) is saved to the `output_dir` within the job's working directory, and Ray Train results are stored at the `ray_storage_path` (`/mnt/user_storage/`).